<a href="https://colab.research.google.com/github/sallew-glitch/brain-tumor-mri-gradcam/blob/compareModels/notebooks/colab_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Brain tumor MRI: model comparison on a free Colab GPU

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU -> Save`.
2. Upload `processed.zip` (made on your laptop, see `docs/COLAB.md`) to Google Drive at `My Drive/btmri/processed.zip`.

Then `Runtime -> Run all`. All results are saved to `My Drive/btmri/runs`.

**If Colab restarts or disconnects** (it forgets everything when that happens): run the **setup cell** again, then continue from where you were. Re-running is always safe: models that already finished are skipped.

In [2]:
# SETUP: safe to re-run at any time. Re-run it after every restart or disconnect.
import glob
import os

from google.colab import drive

DRIVE = "/content/drive/MyDrive/btmri"
RUNS = f"{DRIVE}/runs"
REPO = "https://github.com/sallew-glitch/brain-tumor-mri-gradcam.git"

drive.mount("/content/drive")  # a pop-up asks for permission the first time

if not os.path.exists("/content/project"):
    !git clone -q {REPO} /content/project
%cd /content/project
!git pull -q
!pip install -q -e ".[dev]"

if not os.path.exists("data/processed/metadata.csv"):
    !mkdir -p data && unzip -q -o {DRIVE}/processed.zip -d data/
print("images:", len(os.listdir("data/processed/images")), "(expected 3064)")

!nvidia-smi --query-gpu=name --format=csv,noheader
!python -m pytest -q

Mounted at /content/drive
/content/project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 32.3 MB/s eta 0:00:00
  Building editable for btmri (pyproject.toml) ... done
images: 3064 (expected 3064)
Tesla T4
................................                                         [100%]
32 passed in 16.95s


In [3]:
# TRAIN every model on fold 1 (roughly 20-40 minutes in total on a T4).
# Models that already have a finished fold-1 run in Drive are skipped.
# resnet18_cpu is the laptop baseline config; it runs fine on a GPU too.
MODELS = ["resnet18_cpu", "efficientnet_b0", "densenet121", "resnet50"]

for model in MODELS:
    if glob.glob(f"{RUNS}/{model}_*/fold_1/test_metrics.json"):
        print(f"{model}: already trained, skipping")
        continue
    !python scripts/train.py --config configs/{model}.yaml --folds 1 --set output_dir={RUNS}

resnet18_cpu: already trained, skipping
efficientnet_b0: already trained, skipping
densenet121: already trained, skipping
resnet50: already trained, skipping


In [4]:
# GRAD-CAM on the VALIDATION fold at both layers, for every finished run.
for run in sorted(glob.glob(f"{RUNS}/*/summary.json")):
    run_dir = os.path.dirname(run)
    !python scripts/explain.py --run-dir "{run_dir}" --split val --layers last penultimate

Grad-CAM fold 1 (val, last): 100% 22/22 [00:08<00:00,  2.62it/s]
19:16:49 | INFO | btmri.explain.runner | Fold 1 Grad-CAM (val, last) | pointing game 0.299 | energy in tumor 0.096 (4.5x chance) | IoU 0.165
Grad-CAM fold 1 (val, penultimate): 100% 22/22 [00:05<00:00,  3.86it/s]
19:16:59 | INFO | btmri.explain.runner | Fold 1 Grad-CAM (val, features.denseblock3) | pointing game 0.323 | energy in tumor 0.126 (5.8x chance) | IoU 0.188

Done. See /content/drive/MyDrive/btmri/runs/densenet121_20261013-174107/fold_*/gradcam_val_*/
Grad-CAM fold 1 (val, last): 100% 22/22 [00:04<00:00,  4.48it/s]
19:17:20 | INFO | btmri.explain.runner | Fold 1 Grad-CAM (val, last) | pointing game 0.214 | energy in tumor 0.071 (3.2x chance) | IoU 0.130
Grad-CAM fold 1 (val, penultimate): 100% 22/22 [00:05<00:00,  3.91it/s]
19:17:28 | INFO | btmri.explain.runner | Fold 1 Grad-CAM (val, blocks.4) | pointing game 0.093 | energy in tumor 0.034 (1.6x chance) | IoU 0.042

Done. See /content/drive/MyDrive/btmri/runs/ef

In [5]:
# COMPARE. Choose using the VAL_macro_f1 column and the Grad-CAM table (val split).
!python scripts/compare_runs.py --runs-dir {RUNS}

CLASSIFICATION
                            run           model  folds VAL_macro_f1     test_accuracy test_balanced_accuracy     test_macro_f1    test_macro_auc
    densenet121_20261013-174107     densenet121      1       0.9483 0.9686 +/- 0.0000      0.9672 +/- 0.0000 0.9636 +/- 0.0000 0.9981 +/- 0.0000
efficientnet_b0_20261013-173844 efficientnet_b0      1       0.8995 0.9207 +/- 0.0000      0.9147 +/- 0.0000 0.9098 +/- 0.0000 0.9902 +/- 0.0000
   resnet18_cpu_20261013-173607        resnet18      1       0.9355 0.9391 +/- 0.0000      0.9429 +/- 0.0000 0.9296 +/- 0.0000 0.9938 +/- 0.0000
       resnet50_20261013-174648        resnet50      1       0.9420 0.9244 +/- 0.0000      0.9266 +/- 0.0000 0.9115 +/- 0.0000 0.9931 +/- 0.0000

GRAD-CAM LOCALIZATION (higher is better)
                            run           model split                layer  folds  pointing_hit  energy_ratio_vs_chance   iou
    densenet121_20261013-174107     densenet121   val features.denseblock3      1         0.

In [6]:
for model in ["densenet121", "resnet50", "resnet18_cpu"]:
    !python scripts/train.py --config configs/{model}.yaml --set output_dir={RUNS} name={model}_cv5


19:27:15 | INFO | btmri.training.runner | Run directory: /content/drive/MyDrive/btmri/runs/densenet121_cv5_20261013-192715
19:27:15 | INFO | btmri.training.runner | Fold 1 on cuda | train 1843 imgs (142 patients) | val 679 (fold 2) | test 542
19:27:16 | INFO | timm.models._builder | Loading pretrained weights from Hugging Face hub (timm/densenet121.ra_in1k)
19:27:16 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
19:27:16 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/timm/densenet121.ra_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
19:27:16 | WARNING | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
19:27:16 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/timm/densenet121.ra_in1k/xet-read-token/92007b6200e0b4a4fe68cb4e3947022a928aaaae "HTTP/1.1 200 OK"

model.safetensors: down

In [7]:
for run in sorted(glob.glob(f"{RUNS}/*_cv5_*/summary.json")):
    run_dir = os.path.dirname(run)
    !python scripts/explain.py --run-dir "{run_dir}" --split test --layers last penultimate
!python scripts/compare_runs.py --runs-dir {RUNS}


Grad-CAM fold 1 (test, last): 100% 17/17 [00:05<00:00,  3.07it/s]
20:29:33 | INFO | btmri.explain.runner | Fold 1 Grad-CAM (test, last) | pointing game 0.231 | energy in tumor 0.061 (4.2x chance) | IoU 0.103
Grad-CAM fold 1 (test, penultimate): 100% 17/17 [00:04<00:00,  3.91it/s]
20:29:40 | INFO | btmri.explain.runner | Fold 1 Grad-CAM (test, features.denseblock3) | pointing game 0.216 | energy in tumor 0.080 (5.4x chance) | IoU 0.118
Grad-CAM fold 2 (test, last): 100% 22/22 [00:05<00:00,  3.68it/s]
20:29:49 | INFO | btmri.explain.runner | Fold 2 Grad-CAM (test, last) | pointing game 0.202 | energy in tumor 0.079 (3.6x chance) | IoU 0.124
Grad-CAM fold 2 (test, penultimate): 100% 22/22 [00:05<00:00,  3.92it/s]
20:29:58 | INFO | btmri.explain.runner | Fold 2 Grad-CAM (test, features.denseblock3) | pointing game 0.237 | energy in tumor 0.094 (4.3x chance) | IoU 0.137
Grad-CAM fold 3 (test, last): 100% 18/18 [00:04<00:00,  3.94it/s]
20:30:06 | INFO | btmri.explain.runner | Fold 3 Grad-CAM

## After choosing a model

Run the full 5-fold cross-validation for the chosen model (edit the name), then the final Grad-CAM report on the test folds at the layer you picked:

```python
!python scripts/train.py --config configs/efficientnet_b0.yaml --set output_dir={RUNS}
!python scripts/explain.py --run-dir "{RUNS}/<new_run_name>" --split test --layers penultimate
```

To look at a heatmap image here:

```python
from IPython.display import Image
Image(f"{RUNS}/<run_name>/fold_1/gradcam_val_last/examples.png")
```